# 4: Run CLM with FATES

## FATES

An important component of representing the land surface is accurately capturing vegetation dynamics and their impact on and interaction with the Earth system. Ecosystem demography models explicitly represent the size structure and successional state of vegetation, through direct simulation of plant growth, mortality, and regeneration. Thus, important vegetation characteristics such as vegetation canopy height, succession, and even potential biome shifts become emergent properties of the model rather than being prescribed.

**FATES** is the "**F**unctionally **A**ssembled **T**errestrial **E**cosystem **S**imulator". FATES is a cohort model of vegetation competition and co-existence, allowing a representation of the biosphere which accounts for the division of the land surface into successional stages, and for competition for light between height structured cohorts of representative trees of various plant functional types. Individual plants within FATES are grouped into "cohorts" of the same size and PFT, and these cohorts compete for light and resources on individual "patches" that represent different disturbance histories. This type of ecosystem heterogeneity is in contrast to the default vegetation model in CLM, which uses two (sunlit & shaded) "big leaf" canopies per PFT, each on their own patch, with no representation of within-canopy structural heterogeneity or disturbance history.

When CLM is coupled to FATES ("CLM-FATES"), CLM provides site and soil conditions and atmospheric forcing, while FATES simulates plant physiological, vegetation demography, and biogeochemical processes.

<div>
<img src="../../../images/challenge/FATES_schematic.png" width="700" alt="Conceptual relationship between CLM and CLM-FATES"/>
</div>

*<p style="text-align: left;">Processes simulated in CLM-FATES by each model. Top: processes simulated by FATES when connected to CLM. Arrows in purple indicate conditions supplied to FATES by CLM. Arrows in green indicate conditions supplied to CLM by FATES. Bottom: Processes simulated by CLM when connected to FATES. Green starred variables are simulated and provided by FATES or in the case of aerodynamic resistance (r<sub>a</sub>) are influenced by the FATES-provided roughness length, displacement height, and leaf dimension. †: Only used in FATES hydraulics mode. From Foster et al. (202).</p>*

### FATES Complexity Modes

Currently, FATES can be run in several different "complexity modes", where parts of the vegetation model are driven by input data rather than simulated. These modes can be used to facilitate calibration, test features, or run simulations more quickly. These modes are:

1. **Satellite Phenology** (SP) mode: this mode is designed to run with leaf area index (LAI), stem area index (SAI), and canopy height (HTOP) as input to the model. As such, all processes that are normally used to calculate these values are turned off (e.g., mortality, allocation, etc.)

2. **No-Competition Mode**: this mode runs with full complexity in terms of processes, but places each FATES PFT on its own patch. As such, PFTs do not compete with one another. 

3. **Fixed Biogeography Mode**: this mode turns off prognostic spatial changes in the distribution of vegetation and instead, the model uses input data to determine which PFTs are present at any given gridcell. The patch area for each PFT is derived from the input CLM surface dataset. However, please note that the PFTs in the FATES parameter file do not always map one-to-one with the CLM PFTs on the surface dataset. See the FATES parameter *fates_hlm_pft_map* on the FATES parameter file for the correct mapping of FATES to CLM PFTs.

4. **Full FATES Mode**: All processes are turned on an PFTs are allowed to grow anywhere.

Note that there are different combinations of no-competition and fixed biogeography mode that will result in different model behaviors. See the <a href="https://fates-users-guide.readthedocs.io/en/latest/user/namelist-options.html">FATES namelist documentation</a> for these options.

<div>
<img src="../../../images/challenge/FATES_complexity_modes.png" width="1000px" alt="FATES complexity modes"/>
</div>


We will start by running in Satellite Phenology mode, like in our big-leaf CLM exercises.

<div class="alert alert-info">
<strong>Exercise: Run a FATES case</strong><br><br>
 
Create a case called **i.fates.year1.a** using the compset `I2000Clm60FatesCrujraRs` at `f19_g17` resolution. 

We will need to turn on FATES satellite phenology mode using namelist changes to the `user_nl_clm` file, setting `use_fates_sp`, `use_fates_nocomp`, and `use_fates_fixed_biogeog` to true.

Set the run length to **1 year**. 

Build and run the model.
    
</div>

<div class="alert alert-warning">  
<details>

<summary> <font face="Times New Roman" color='blue'>Click here for hints</font> </summary>

<br>

**How do I compile?**

You can compile with the command:
```
qcmd -- ./case.build
```

<br>

**How do I turn on FATES SP mode?**

Use namelist variables: ``use_fates_sp``, ``use_fates_nocomp``, ``use_fates_fixed_biogeog``. 

Look at the <a href="https://fates-users-guide.readthedocs.io/en/latest/user/namelist-options.html">online documentation</a>  for these variables.

<br>

**How do I change the run length to 1 year?**

Use xml variables: ``STOP_OPTION`` and ``STOP_N``. 

<br>

**How do I check my solution?**

When your run is completed, go to the archive directory and navigate to the subdirectory `lnd/hist`

```
cd /glade/derecho/scratch/$USER/archive/i.fates.year1.a
cd lnd/hist
```

(1) Check that your archive directory contains the files:

```
i.fates.year1.a.clm2.h0a.0001-01.nc
```

(2) Investigate the contents of one of the files with ``ncdump``.

```
ncdump -h b1850_high_freq.cam.h1.0001-01-01-00000.nc
```

(3) Check the number of timesteps in the ``h1`` and the ``h2`` files.
Look at the sizes of the files.

</details>
</div>

<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

**Create a new case**

Create a new case <font face="Courier" color='purple'><strong>i.fates.year1.a </strong></font> with the command:
```
cd /glade/u/home/$USER/code/my_cesm_code/cime/scripts
./create_newcase --case ~/cases/i.fates.year1.a --compset I2000Clm60FatesCrujraRs --res f19_g17 --run-unsupported
```
<br>

**Setup**
    
Invoke case.setup with the command:
    
```    
cd ~/cases/i.fates.year1.a 
./case.setup
```
<br>


**Customize namelist**
    
Edit the file `user_nl_clm` and add the lines:
```
use_fates_sp = .true.
use_fates_nocomp = .true.
use_fates_fixed_biogeog = .true.
```

Check the namelist by running:
```   
./preview_namelists
```
<br>

**Set run length**
    
Change the `run length`:
```   
./xmlchange STOP_N=1,STOP_OPTION=nyears
```

<br>

**Change the job queue and account number**

If needed, change `job queue` and `account number`. <br>
For instance, to run in the queue `tutorial` and the project number ``UESM0015`` (You should use the project number given for this tutorial), use the command:
```  
./xmlchange JOB_QUEUE=tutorial,PROJECT=UESM0015 --force
```

<br>

**Change the runtime length**

Change wallclock time to use the maximum of the allowed wallclock on Derecho:
```
./xmlchange --subgroup case.run JOB_WALLCLOCK_TIME=12:00:00
```
<br>

**Build case:**
```
qcmd -- ./case.build
```
<br>
    
    
**Submit case:**
```
./case.submit
```
<br>

**Check the run:**
When the run is completed, look into the archive directory for: 
<font face="Courier" color='purple'><strong>i.fates.year1.a</strong></font>.  
    
(1) Check that your archive directory on derecho (The path will be different on other machines): 
```
cd /glade/derecho/scratch/$USER/archive/i.fates.year1.a/lnd/hist

ls 
```
<br>

(2) Compare to control run:
```
ncdiff i.day5.a_pft.clm2.XXX.nc /glade/derecho/scratch/$USER/archive/i.day5.a/lnd/hist/i.day5.a.clm2.XXX.nc i_diff.nc

ncview i_diff.nc
```

</details>
</div>

## Test your understanding

- How did rholvis change (increase/decrease)? Given this, what do you expect the model response to be?
- What changes do you see from the control case with the modified rholvis parameter?
- ... OTHERS?  